In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        pass

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# ============================================================================
# CELL 1 : IMPORTS, PATHS & HELPER TO LOAD IMAGES
# ============================================================================
import os, time, warnings
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import cv2
from scipy import stats
import pywt
from skimage.feature import graycomatrix

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

# ---------- EDIT THIS PATH ----------
CROPPED_DIR = '/kaggle/input/datasets/sabitahamedpreanto/moringa-segmented/cropped_leaves'
OUTPUT_DIR  = '/kaggle/working/feature_analysis'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Class names (must match the subfolders you used earlier)
CLASSES = ['Healthy', 'Yellow', 'Cercospora Leaf Spot', 'Bacterial Leaf Spot']
class_to_idx = {c:i for i,c in enumerate(CLASSES)}

# Target image size (must match your cropped images)
IMG_SIZE = 256

print("Libraries loaded. Output directory:", OUTPUT_DIR)

In [ ]:
# ============================================================================
# CELL 2 : FEATURE EXTRACTION HELPER FUNCTIONS
# ============================================================================

def compute_14_features(mat: np.ndarray) -> np.ndarray:
    """
    Compute 14 statistical features from a 2D array.
    Features: area, mean, std, energy, median, skewness, entropy,
              min, max, mean_abs_dev, kurtosis, range, rms, uniformity.
    """
    mat = mat.ravel().astype(np.float64)
    area = len(mat)
    mean_val = np.mean(mat)
    std_val = np.std(mat)
    energy = np.sum(mat**2)
    median_val = np.median(mat)
    skewness = stats.skew(mat)
    kurtosis = stats.kurtosis(mat)
    min_val = np.min(mat)
    max_val = np.max(mat)
    range_val = max_val - min_val
    mad = np.mean(np.abs(mat - mean_val))
    rms = np.sqrt(np.mean(mat**2))

    # Entropy and uniformity via histogram (256 bins)
    counts, _ = np.histogram(mat, bins=256, range=(min_val, max_val))
    # If all values are identical, set uniform probabilities
    if max_val == min_val:
        probs = np.ones(256) / 256
    else:
        probs = counts / area
    probs += 1e-12  # avoid log(0)
    entropy_val = -np.sum(probs * np.log2(probs))
    uniformity = np.sum(probs**2)

    return np.array([area, mean_val, std_val, energy, median_val,
                     skewness, entropy_val, min_val, max_val, mad,
                     kurtosis, range_val, rms, uniformity])

def extract_glcm_features(gray_img):
    """GLCM with 14 features per orientation → 4*14 = 56 features."""
    glcm = graycomatrix(gray_img, distances=[1], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
                        levels=256, symmetric=True, normed=True)
    feats = []
    for a in range(4):
        mat = glcm[:, :, 0, a]
        feats.append(compute_14_features(mat))
    return np.concatenate(feats)

def extract_gldm_features(gray_img):
    """GLDM with 14 features per orientation → 56 features."""
    rows, cols = gray_img.shape
    offsets = [(1,0), (1,1), (0,1), (-1,1)]  # 0°, 45°, 90°, 135°
    all_feats = []
    for dx, dy in offsets:
        # Create difference matrix (size 256 x 256)
        diff_mat = np.zeros((256, 256), dtype=np.float64)
        for i in range(rows):
            for j in range(cols):
                val = gray_img[i, j]
                ni, nj = i+dx, j+dy
                if 0 <= ni < rows and 0 <= nj < cols:
                    diff = abs(int(val) - int(gray_img[ni, nj]))
                    diff_mat[val, diff] += 1
        # Normalize
        total = diff_mat.sum()
        if total > 0:
            diff_mat /= total
        all_feats.append(compute_14_features(diff_mat))
    return np.concatenate(all_feats)

def extract_fft_features(gray_img):
    """FFT magnitude → 14 features."""
    f = np.fft.fft2(gray_img)
    fshift = np.fft.fftshift(f)
    mag = np.abs(fshift)
    return compute_14_features(mag)

def extract_dwt_features(gray_img, wavelet='db2'):
    """2-level DWT → 8 sub-bands → 8*14 = 112 features."""
    # Level 1
    LL1, (LH1, HL1, HH1) = pywt.dwt2(gray_img, wavelet)
    # Level 2 (on LL1)
    LL2, (LH2, HL2, HH2) = pywt.dwt2(LL1, wavelet)
    bands = [LL1, LH1, HL1, HH1, LL2, LH2, HL2, HH2]
    feats = [compute_14_features(b) for b in bands]
    return np.concatenate(feats)

def extract_all_features(rgb_img):
    """
    Given a 256x256 RGB image, compute the full 252-dimensional feature vector.
    """
    gray = cv2.cvtColor(rgb_img, cv2.COLOR_RGB2GRAY)
    # Texture (direct from image) -> 14
    tex_feat = compute_14_features(gray)
    # GLCM -> 56
    glcm_feat = extract_glcm_features(gray)
    # GLDM -> 56
    gldm_feat = extract_gldm_features(gray)
    # FFT -> 14
    fft_feat = extract_fft_features(gray)
    # DWT -> 112
    dwt_feat = extract_dwt_features(gray)
    return np.concatenate([tex_feat, glcm_feat, gldm_feat, fft_feat, dwt_feat])

print("Feature extraction functions defined.")

In [ ]:
# ============================================================================
# CELL 3 (UPDATED) : COMPUTE 252 FEATURES FROM SEGMENTED (OVERLAY) IMAGES
# ============================================================================

import os
import numpy as np
import pandas as pd
import cv2
from tqdm.notebook import tqdm

# ---------- Important: use OVERLAY_DIR, not CROPPED_DIR ----------
OVERLAY_DIR = '/kaggle/input/datasets/sabitahamedpreanto/moringa-segmented/overlays'   # the blended images with disease masks
OUTPUT_DIR  = '/kaggle/working/feature_analysis'
os.makedirs(OUTPUT_DIR, exist_ok=True)

CLASSES = ['Healthy', 'Yellow', 'Cercospora Leaf Spot', 'Bacterial Leaf Spot']
class_to_idx = {c: i for i, c in enumerate(CLASSES)}

# Collect overlay image paths
image_paths = []
labels = []
for cls_name in CLASSES:
    for fname in os.listdir(OVERLAY_DIR):
        if fname.startswith(cls_name + '__'):
            image_paths.append(os.path.join(OVERLAY_DIR, fname))
            labels.append(class_to_idx[cls_name])

print(f"Found {len(image_paths)} overlay images.")

# Extract 252 features from overlay images
feature_list = []
for impath in tqdm(image_paths, desc="Extracting features from overlays"):
    img_bgr = cv2.imread(impath)
    if img_bgr is None:
        print(f"Warning: cannot read {impath}, filling with zeros")
        feature_list.append(np.zeros(252))
        continue
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    feat = extract_all_features(img_rgb)   # uses the same helper functions from Cell 2
    feature_list.append(feat)

X = np.array(feature_list)
y = np.array(labels)

# Build DataFrame with column names (as before)
col_names = (
    ['TEX_' + str(i) for i in range(14)] +
    ['GLCM_' + str(i) for i in range(56)] +
    ['GLDM_' + str(i) for i in range(56)] +
    ['FFT_' + str(i) for i in range(14)] +
    ['DWT_' + str(i) for i in range(112)]
)
df = pd.DataFrame(X, columns=col_names)
df['label'] = y
df['image_path'] = image_paths

csv_path = os.path.join(OUTPUT_DIR, 'features_252_from_overlays.csv')
df.to_csv(csv_path, index=False)
print(f"Feature matrix (from overlays) saved to {csv_path}, shape = {X.shape}")

In [ ]:
# ============================================================================
# CELL 4 : PCA (→70) & KPCA (→65) on the 252 features
# ============================================================================
from sklearn.decomposition import PCA, KernelPCA
from sklearn.preprocessing import StandardScaler

# Load data
#df = pd.read_csv(os.path.join(OUTPUT_DIR, 'features_252_from_overlays.csv'))
df = pd.read_csv('/kaggle/input/models/sabitahamedpreanto/faetures/other/default/1/features_252_from_overlays.csv')
X = df.drop(columns=['label', 'image_path']).values
y = df['label'].values

# Standardize features (important for PCA/KPCA)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ----- PCA (70 components) -----
pca = PCA(n_components=70, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print(f"PCA explained variance ratio (first 5): {pca.explained_variance_ratio_[:5]}")

df_pca = pd.DataFrame(X_pca, columns=[f'PC{i+1}' for i in range(70)])
df_pca['label'] = y
pca_path = '/kaggle/working/reduced_pca_70.csv'
df_pca.to_csv(pca_path, index=False)
print(f"PCA-reduced features saved to {pca_path}")

# ----- KPCA (RBF kernel, 65 components) -----
kpca = KernelPCA(n_components=65, kernel='rbf', random_state=42, n_jobs=-1)
X_kpca = kpca.fit_transform(X_scaled)

df_kpca = pd.DataFrame(X_kpca, columns=[f'KPC{i+1}' for i in range(65)])
df_kpca['label'] = y
#kpca_path = os.path.join(OUTPUT_DIR, 'reduced_kpca_65.csv')
kpca_path = '/kaggle/working/reduced_pca_70.csv'
df_kpca.to_csv(kpca_path, index=False)
print(f"KPCA-reduced features saved to {kpca_path}")

In [ ]:
# ============================================================================
# CELL 5 : SPARSE AUTOENCODER (252 → 60 → 252)
# ============================================================================
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from sklearn.preprocessing import MinMaxScaler

# Reload data
#df = pd.read_csv(os.path.join(OUTPUT_DIR, 'features_252_from_overlays.csv'))
df = pd.read_csv('/kaggle/input/models/sabitahamedpreanto/faetures/other/default/1/features_252_from_overlays.csv')
X = df.drop(columns=['label', 'image_path']).values

# Normalize to [0,1] for sigmoid activation
mms = MinMaxScaler()
X_norm = mms.fit_transform(X)

# Build Sparse AE
input_dim = X.shape[1]  # 252
encoding_dim = 60

# Encoder
input_layer = layers.Input(shape=(input_dim,))
encoded = layers.Dense(encoding_dim, activation='sigmoid',
                       activity_regularizer=regularizers.l1(1e-5))(input_layer)
# Decoder
decoded = layers.Dense(input_dim, activation='sigmoid')(encoded)

autoencoder = models.Model(input_layer, decoded)
autoencoder.compile(optimizer='adam', loss='mse')  # MSE for reconstruction

# Custom sparsity is partly handled by activity_regularizer, which approximates KL penalty.
# For a proper sparsity constraint you would add a custom loss, but L1 activity
# regularizer encourages very low activations (close to 0), simulating sparsity.
# This is a practical simplification; final experiments can use a custom KL loss.

# Train
history = autoencoder.fit(X_norm, X_norm,
                          epochs=50,
                          batch_size=32,
                          shuffle=True,
                          validation_split=0.1,
                          verbose=0)
print("Sparse AE training complete. Last val loss:", history.history['val_loss'][-1])

# Extract encoder
encoder = models.Model(input_layer, encoded)
X_sparse_ae = encoder.predict(X_norm)

df_sae = pd.DataFrame(X_sparse_ae, columns=[f'SAE_{i+1}' for i in range(encoding_dim)])
df_sae['label'] = df['label'].values
sae_path = '/kaggle/working/reduced_sparse_ae_60.csv'
df_sae.to_csv(sae_path, index=False)
print(f"Sparse AE features saved to {sae_path}")

In [ ]:
# ============================================================================
# CELL 6 : STACKED AUTOENCODER (252 → 180 → 126 → 180 → 252)
# ============================================================================
import tensorflow as tf
from tensorflow.keras import layers, models

# Reload data (already normalized from previous cell? Re-create to be safe)
#df = pd.read_csv(os.path.join(OUTPUT_DIR, 'features_252_from_overlays.csv'))
df = pd.read_csv('/kaggle/input/models/sabitahamedpreanto/faetures/other/default/1/features_252_from_overlays.csv')
X = df.drop(columns=['label', 'image_path']).values
mms = MinMaxScaler()
X_norm = mms.fit_transform(X)

input_dim = X.shape[1]
bottleneck_dim = 126

# Encoder
input_layer = layers.Input(shape=(input_dim,))
encoded1 = layers.Dense(180, activation='relu')(input_layer)
encoded2 = layers.Dense(bottleneck_dim, activation='relu')(encoded1)

# Decoder
decoded1 = layers.Dense(180, activation='relu')(encoded2)
decoded2 = layers.Dense(input_dim, activation='sigmoid')(decoded1)

stacked_ae = models.Model(input_layer, decoded2)
stacked_ae.compile(optimizer='adam', loss='mse')

history = stacked_ae.fit(X_norm, X_norm,
                         epochs=50,
                         batch_size=32,
                         shuffle=True,
                         validation_split=0.1,
                         verbose=0)
print("Stacked AE training complete. Last val loss:", history.history['val_loss'][-1])

# Bottleneck encoder part
bottleneck_model = models.Model(input_layer, encoded2)
X_stacked_ae = bottleneck_model.predict(X_norm)

df_sae2 = pd.DataFrame(X_stacked_ae, columns=[f'StackedAE_{i+1}' for i in range(bottleneck_dim)])
df_sae2['label'] = df['label'].values
#stacked_path = os.path.join(OUTPUT_DIR, 'reduced_stacked_ae_126.csv')
staked_path = '/kaggle/working/reduced_stacked_ae_126.csv'
df_sae2.to_csv(stacked_path, index=False)
print(f"Stacked AE features saved to {stacked_path}")

# Custom Model

# CNN + Handcrafted Feature Fusion

Improved version of V1

# NOVEL + Highest Accuracy (89%)

# Run this code without Early Stooper

In [ ]:
# ============================================================================
# CELL: NOVEL LIGHTWEIGHT HYBRID MODEL (CNN + Handcrafted Features Fusion)
#       with SE attention & Gated Cross-Modal Fusion
#       Outputs: accuracy/loss curves, CM, classification report, ROC curves
#       (EarlyStopping removed - full 150 epochs)
# ============================================================================
import os, numpy as np, pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, auc)
import matplotlib.pyplot as plt, seaborn as sns

# ---------- CONFIG ----------
OVERLAY_DIR   = '/kaggle/input/datasets/sabitahamedpreanto/moringa-segmented/overlays'
FEATURES_CSV  = '/kaggle/input/models/sabitahamedpreanto/faetures/other/default/1/features_252_from_overlays.csv'
OUTPUT_MODEL  = '/kaggle/working/hybrid_fusion_model.h5'
IMG_SIZE      = 256
BATCH_SIZE    = 32
EPOCHS        = 150                   # full 150 epochs
RANDOM_SEED   = 42
CLASS_NAMES   = ['Healthy', 'Yellow', 'Cercospora Leaf Spot', 'Bacterial Leaf Spot']
NUM_CLASSES   = 4
LABEL_SMOOTH  = 0.03

# ---------- 1. Load features & labels ----------
df = pd.read_csv(FEATURES_CSV)
feature_cols = [c for c in df.columns if c not in ['label', 'image_path']]
X_hand = df[feature_cols].values.astype(np.float32)
y      = df['label'].values.astype(np.int32)
paths  = df['image_path'].values

# Train/test split (80/20)
indices = np.arange(len(y))
train_idx, test_idx = train_test_split(indices, test_size=0.2, stratify=y,
                                       random_state=RANDOM_SEED)

# Standardize
scaler = StandardScaler()
X_hand_train = scaler.fit_transform(X_hand[train_idx])
X_hand_test  = scaler.transform(X_hand[test_idx])
y_train = y[train_idx]
y_test  = y[test_idx]
paths_train = paths[train_idx]
paths_test  = paths[test_idx]

# Further split train into train/val (80/20)
train_idx2, val_idx2 = train_test_split(np.arange(len(y_train)), test_size=0.2,
                                        stratify=y_train, random_state=RANDOM_SEED)
X_hand_train2 = X_hand_train[train_idx2]
X_hand_val2   = X_hand_train[val_idx2]
y_train2 = y_train[train_idx2]
y_val2   = y_train[val_idx2]
paths_train2 = paths_train[train_idx2]
paths_val2   = paths_train[val_idx2]

# ---------- 2. Data augmentation ----------
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
], name='data_aug')

# ---------- 3. Image loader ----------
def load_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0
    return img

# ---------- 4. tf.data pipelines ----------
def create_hybrid_dataset(paths, feat_array, labels, training=False):
    path_ds = tf.data.Dataset.from_tensor_slices(paths)
    feat_ds = tf.data.Dataset.from_tensor_slices(feat_array)
    label_ds = tf.data.Dataset.from_tensor_slices(labels)
    ds = tf.data.Dataset.zip((path_ds, feat_ds, label_ds))

    def preprocess(path, feat, label):
        img = load_image(path)
        if training:
            img = data_augmentation(img, training=True)
        return (img, feat), label

    ds = ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.shuffle(1024)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = create_hybrid_dataset(paths_train2, X_hand_train2, y_train2, training=True)
val_ds   = create_hybrid_dataset(paths_val2, X_hand_val2, y_val2, training=False)
test_ds  = create_hybrid_dataset(paths_test, X_hand_test, y_test, training=False)

# ---------- 5. Model architecture ----------
def se_block(x, ratio=16):
    filters = x.shape[-1]
    se = layers.GlobalAveragePooling2D()(x)
    se = layers.Dense(filters // ratio, activation='relu')(se)
    se = layers.Dense(filters, activation='sigmoid')(se)
    return layers.Multiply()([x, se])

# CNN stem
img_input = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='image_input')
x = layers.Conv2D(32, (3,3), activation='relu', padding='same')(img_input)
x = layers.MaxPooling2D((2,2))(x)
x = layers.BatchNormalization()(x)
x = se_block(x)

x = layers.Conv2D(64, (3,3), activation='relu', padding='same')(x)
x = layers.MaxPooling2D((2,2))(x)
x = layers.BatchNormalization()(x)
x = se_block(x)

x = layers.Conv2D(128, (3,3), activation='relu', padding='same')(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu', name='img_features')(x)
x = layers.Dropout(0.3)(x)

# Handcrafted input
feat_input = layers.Input(shape=(252,), name='feat_input')

# ---- Gated Cross‑Modal Fusion ----
img_gate_proj = layers.Dense(252, activation='sigmoid')(x)      # image → gate for handcrafted
gated_feat = layers.Multiply()([feat_input, img_gate_proj])

feat_gate_proj = layers.Dense(128, activation='sigmoid')(feat_input)  # handcrafted → gate for image
gated_img_feat = layers.Multiply()([x, feat_gate_proj])

concat = layers.Concatenate()([gated_img_feat, gated_feat])

# Fusion head
z = layers.Dense(128, activation='relu')(concat)
z = layers.BatchNormalization()(z)
z = layers.Dropout(0.4)(z)
z = layers.Dense(64, activation='relu')(z)
z = layers.BatchNormalization()(z)
z = layers.Dropout(0.4)(z)

output = layers.Dense(NUM_CLASSES, activation='softmax')(z)
model = models.Model(inputs=[img_input, feat_input], outputs=output)

# ---------- 6. Custom loss with label smoothing ----------
def smooth_loss(y_true, y_pred, smoothing=LABEL_SMOOTH):
    num_classes = tf.shape(y_pred)[-1]
    y_true_one_hot = tf.one_hot(tf.cast(y_true, tf.int32), depth=num_classes,
                                on_value=1.0 - smoothing,
                                off_value=smoothing / (tf.cast(num_classes, tf.float32) - 1.0))
    y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
    return -tf.reduce_mean(tf.reduce_sum(y_true_one_hot * tf.math.log(y_pred), axis=-1))

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=smooth_loss,
    metrics=['accuracy']
)
model.summary()

# ---------- 7. Callbacks (only reduce_lr, no early stopping) ----------
reduce_lr = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                        patience=5, min_lr=1e-6)

# ---------- 8. Train for full 150 epochs ----------
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[reduce_lr],
    verbose=1
)

# ---------- 9. Acc/Loss curves ----------
def plot_training_curves(hist):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(hist.history['accuracy'], label='Train')
    ax1.plot(hist.history['val_accuracy'], label='Validation')
    ax1.set_title('Accuracy over Epochs')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax2.plot(hist.history['loss'], label='Train')
    ax2.plot(hist.history['val_loss'], label='Validation')
    ax2.set_title('Loss over Epochs')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
    ax2.legend()
    plt.tight_layout()
    plt.savefig('/kaggle/working/accuracy_loss_curves.png', dpi=150)
    plt.close()

plot_training_curves(history)

# ---------- 10. Evaluation on test set ----------
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"\n✅ Hybrid Fusion Model Test Accuracy: {test_acc:.4f}")

y_pred_probs = model.predict(test_ds)
y_pred = np.argmax(y_pred_probs, axis=1)

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=4))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix – Enhanced Hybrid Model')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.tight_layout()
plt.savefig('/kaggle/working/hybrid_fusion_cm.png', dpi=150)
plt.close()

# ---------- 11. ROC curves (per class) ----------
y_test_bin = label_binarize(y_test, classes=range(NUM_CLASSES))
fpr = dict(); tpr = dict(); roc_auc = dict()
for i in range(NUM_CLASSES):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_pred_probs[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Plot all ROC curves
plt.figure(figsize=(8,6))
for i in range(NUM_CLASSES):
    plt.plot(fpr[i], tpr[i],
             label=f'{CLASS_NAMES[i]} (AUC = {roc_auc[i]:.3f})')
plt.plot([0,1], [0,1], 'k--')
plt.xlim([0.0, 1.0]); plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves per Class')
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig('/kaggle/working/roc_curves.png', dpi=150)
plt.close()

# ---------- 12. Save model ----------
model.save(OUTPUT_MODEL)
print(f"Model saved to {OUTPUT_MODEL}")

In [ ]:
# ============================================================================
# FINAL HYBRID MODEL (SE + Gated Fusion) with MixUp, Cosine LR, L2, EarlyStopping
# Outputs: accuracy/loss curves, CM, classification report, ROC curves
# ============================================================================
import os, numpy as np, pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, regularizers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt, seaborn as sns

# ---------- CONFIG ----------
OVERLAY_DIR   = '/kaggle/input/datasets/sabitahamedpreanto/moringa-segmented/overlays'
FEATURES_CSV  = '/kaggle/input/models/sabitahamedpreanto/faetures/other/default/1/features_252_from_overlays.csv'
OUTPUT_MODEL  = '/kaggle/working/hybrid_fusion_model.h5'
IMG_SIZE      = 256
BATCH_SIZE    = 32
EPOCHS        = 150                     # early stopping will terminate earlier
RANDOM_SEED   = 42
CLASS_NAMES   = ['Healthy', 'Yellow', 'Cercospora Leaf Spot', 'Bacterial Leaf Spot']
NUM_CLASSES   = 4
MIXUP_ALPHA   = 0.2

# ---------- 1. Load features & labels ----------
df = pd.read_csv(FEATURES_CSV)
feature_cols = [c for c in df.columns if c not in ['label', 'image_path']]
X_hand = df[feature_cols].values.astype(np.float32)
y      = df['label'].values.astype(np.int32)
paths  = df['image_path'].values

# Train/test split (80/20)
indices = np.arange(len(y))
train_idx, test_idx = train_test_split(indices, test_size=0.2, stratify=y,
                                       random_state=RANDOM_SEED)

# Standardize
scaler = StandardScaler()
X_hand_train = scaler.fit_transform(X_hand[train_idx])
X_hand_test  = scaler.transform(X_hand[test_idx])
y_train = y[train_idx]
y_test  = y[test_idx]
paths_train = paths[train_idx]
paths_test  = paths[test_idx]

# Further split train into train/val (80/20)
train_idx2, val_idx2 = train_test_split(np.arange(len(y_train)), test_size=0.2,
                                        stratify=y_train, random_state=RANDOM_SEED)
X_hand_train2 = X_hand_train[train_idx2]
X_hand_val2   = X_hand_train[val_idx2]
y_train2 = y_train[train_idx2]
y_val2   = y_train[val_idx2]
paths_train2 = paths_train[train_idx2]
paths_val2   = paths_train[val_idx2]

# ---------- 2. Data augmentation (slightly stronger) ----------
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.15),
    layers.RandomBrightness(0.1),
], name='data_aug')

# ---------- 3. Image loader ----------
def load_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0
    return img

# ---------- 4. tf.data pipelines with MixUp and one-hot labels ----------
def create_hybrid_dataset(paths, feat_array, labels, training=False):
    path_ds = tf.data.Dataset.from_tensor_slices(paths)
    feat_ds = tf.data.Dataset.from_tensor_slices(feat_array)
    label_ds = tf.data.Dataset.from_tensor_slices(labels)
    ds = tf.data.Dataset.zip((path_ds, feat_ds, label_ds))

    def preprocess(path, feat, label):
        img = load_image(path)
        if training:
            img = data_augmentation(img, training=True)
        return (img, feat), label

    ds = ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.shuffle(1024)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

    if training:
        # MixUp augmentation
        def mixup_batch(images, features, labels):
            batch_size = tf.shape(images)[0]
            indices = tf.random.shuffle(tf.range(batch_size))
            # Lam per sample (shape: batch,1,1,1)
            lam = tf.random.uniform(shape=[batch_size, 1, 1, 1], minval=0.0, maxval=1.0)
            lam = tf.maximum(lam, 1 - MIXUP_ALPHA)

            mixed_images = lam * images + (1. - lam) * tf.gather(images, indices)

            lam_feat = lam[:, 0, 0, 0:1]  # (batch,1)
            mixed_features = lam_feat * features + (1. - lam_feat) * tf.gather(features, indices)

            labels_onehot = tf.one_hot(labels, NUM_CLASSES)
            mixed_labels = lam_feat * labels_onehot + (1. - lam_feat) * tf.gather(labels_onehot, indices)
            return (mixed_images, mixed_features), mixed_labels

        ds = ds.map(lambda img_feat, lbl: mixup_batch(img_feat[0], img_feat[1], lbl),
                    num_parallel_calls=tf.data.AUTOTUNE)
    else:
        # One-hot conversion for validation/test (categorical crossentropy)
        ds = ds.map(lambda imgs_feats, lbls: ((imgs_feats[0], imgs_feats[1]), tf.one_hot(lbls, NUM_CLASSES)),
                    num_parallel_calls=tf.data.AUTOTUNE)

    return ds

train_ds = create_hybrid_dataset(paths_train2, X_hand_train2, y_train2, training=True)
val_ds   = create_hybrid_dataset(paths_val2, X_hand_val2, y_val2, training=False)
test_ds  = create_hybrid_dataset(paths_test, X_hand_test, y_test, training=False)

# ---------- 5. Model architecture (SE + Gated Fusion + L2 regularisation) ----------
def se_block(x, ratio=16):
    filters = x.shape[-1]
    se = layers.GlobalAveragePooling2D()(x)
    se = layers.Dense(filters // ratio, activation='relu')(se)
    se = layers.Dense(filters, activation='sigmoid')(se)
    return layers.Multiply()([x, se])

# CNN stem
img_input = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='image_input')
x = layers.Conv2D(32, (3,3), activation='relu', padding='same')(img_input)
x = layers.MaxPooling2D((2,2))(x)
x = layers.BatchNormalization()(x)
x = se_block(x)

x = layers.Conv2D(64, (3,3), activation='relu', padding='same')(x)
x = layers.MaxPooling2D((2,2))(x)
x = layers.BatchNormalization()(x)
x = se_block(x)

x = layers.Conv2D(128, (3,3), activation='relu', padding='same')(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(1e-4), name='img_features')(x)
x = layers.Dropout(0.3)(x)

# Handcrafted input
feat_input = layers.Input(shape=(252,), name='feat_input')

# ---- Gated Cross‑Modal Fusion ----
img_gate_proj = layers.Dense(252, activation='sigmoid')(x)
gated_feat = layers.Multiply()([feat_input, img_gate_proj])

feat_gate_proj = layers.Dense(128, activation='sigmoid')(feat_input)
gated_img_feat = layers.Multiply()([x, feat_gate_proj])

concat = layers.Concatenate()([gated_img_feat, gated_feat])

# Fusion head (stronger dropout, L2)
z = layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(concat)
z = layers.BatchNormalization()(z)
z = layers.Dropout(0.5)(z)
z = layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(z)
z = layers.BatchNormalization()(z)
z = layers.Dropout(0.5)(z)

output = layers.Dense(NUM_CLASSES, activation='softmax')(z)
model = models.Model(inputs=[img_input, feat_input], outputs=output)

# ---------- 6. Cosine decay schedule + compile ----------
steps_per_epoch = int(np.ceil(len(paths_train2) / BATCH_SIZE))
cosine_scheduler = tf.keras.optimizers.schedules.CosineDecayRestarts(
    initial_learning_rate=1e-3,
    first_decay_steps=steps_per_epoch * 20,
    t_mul=2.0,
    m_mul=0.9,
    alpha=1e-6
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=cosine_scheduler),
    loss='categorical_crossentropy',   # works with one-hot labels
    metrics=['accuracy']
)
model.summary()

# ---------- 7. Callbacks (early stopping back) ----------
early_stop = callbacks.EarlyStopping(
    monitor='val_accuracy', patience=20, restore_best_weights=True
)

# ---------- 8. Train ----------
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[early_stop],
    verbose=1
)

# ---------- 9. Acc/Loss curves ----------
def plot_training_curves(hist):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(hist.history['accuracy'], label='Train')
    ax1.plot(hist.history['val_accuracy'], label='Validation')
    ax1.set_title('Accuracy over Epochs')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax2.plot(hist.history['loss'], label='Train')
    ax2.plot(hist.history['val_loss'], label='Validation')
    ax2.set_title('Loss over Epochs')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
    ax2.legend()
    plt.tight_layout()
    plt.savefig('/kaggle/working/accuracy_loss_curves.png', dpi=150)
    plt.close()

plot_training_curves(history)

# ---------- 10. Evaluation on test set ----------
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"\n✅ Hybrid Fusion Model Test Accuracy: {test_acc:.4f}")

y_pred_probs = model.predict(test_ds)
y_pred = np.argmax(y_pred_probs, axis=1)   # back to sparse labels for report

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=4))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix – Final Hybrid Model')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.tight_layout()
plt.savefig('/kaggle/working/hybrid_fusion_cm.png', dpi=150)
plt.close()

# ---------- 11. ROC curves (per class) ----------
y_test_bin = label_binarize(y_test, classes=range(NUM_CLASSES))
fpr = dict(); tpr = dict(); roc_auc = dict()
for i in range(NUM_CLASSES):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_pred_probs[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

plt.figure(figsize=(8,6))
for i in range(NUM_CLASSES):
    plt.plot(fpr[i], tpr[i],
             label=f'{CLASS_NAMES[i]} (AUC = {roc_auc[i]:.3f})')
plt.plot([0,1], [0,1], 'k--')
plt.xlim([0.0, 1.0]); plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves per Class')
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig('/kaggle/working/roc_curves.png', dpi=150)
plt.close()

# ---------- 12. Save model ----------
model.save(OUTPUT_MODEL)
print(f"Model saved to {OUTPUT_MODEL}")

In [ ]:
# ============================================================================
# CELL: NOVEL LIGHTWEIGHT HYBRID MODEL (CNN + Handcrafted Fusion) – V2
#       SE + Gated Cross-Modal Fusion + MixUp + TTA
#       Outputs: accuracy/loss curves, CM, classification report, ROC curves
# ============================================================================
import os, numpy as np, pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, mixed_precision
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt, seaborn as sns

# ---------- CONFIG ----------
OVERLAY_DIR   = '/kaggle/input/datasets/sabitahamedpreanto/moringa-segmented/overlays'
FEATURES_CSV  = '/kaggle/input/models/sabitahamedpreanto/faetures/other/default/1/features_252_from_overlays.csv'
OUTPUT_MODEL  = '/kaggle/working/hybrid_fusion_model_v2.h5'
IMG_SIZE      = 256
BATCH_SIZE    = 32
EPOCHS        = 100                # early stopping will terminate earlier
RANDOM_SEED   = 42
CLASS_NAMES   = ['Healthy', 'Yellow', 'Cercospora Leaf Spot', 'Bacterial Leaf Spot']
NUM_CLASSES   = 4
MIXUP_ALPHA   = 0.2                # MixUp strength

# ---------- 1. Load features & labels ----------
df = pd.read_csv(FEATURES_CSV)
feature_cols = [c for c in df.columns if c not in ['label', 'image_path']]
X_hand = df[feature_cols].values.astype(np.float32)
y      = df['label'].values.astype(np.int32)
paths  = df['image_path'].values

# Train/test split (80/20)
indices = np.arange(len(y))
train_idx, test_idx = train_test_split(indices, test_size=0.2, stratify=y,
                                       random_state=RANDOM_SEED)

# Standardize
scaler = StandardScaler()
X_hand_train = scaler.fit_transform(X_hand[train_idx])
X_hand_test  = scaler.transform(X_hand[test_idx])
y_train = y[train_idx]
y_test  = y[test_idx]
paths_train = paths[train_idx]
paths_test  = paths[test_idx]

# Further split train into train/val (80/20)
train_idx2, val_idx2 = train_test_split(np.arange(len(y_train)), test_size=0.2,
                                        stratify=y_train, random_state=RANDOM_SEED)
X_hand_train2 = X_hand_train[train_idx2]
X_hand_val2   = X_hand_train[val_idx2]
y_train2 = y_train[train_idx2]
y_val2   = y_train[val_idx2]
paths_train2 = paths_train[train_idx2]
paths_val2   = paths_train[val_idx2]

# ---------- 2. Data augmentation (without MixUp) ----------
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.1),
], name='data_aug')

# ---------- 3. Image loader ----------
def load_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0
    return img

# ---------- 4. tf.data pipelines with MixUp ----------
def create_hybrid_dataset(paths, feat_array, labels, training=False):
    path_ds = tf.data.Dataset.from_tensor_slices(paths)
    feat_ds = tf.data.Dataset.from_tensor_slices(feat_array)
    label_ds = tf.data.Dataset.from_tensor_slices(labels)
    ds = tf.data.Dataset.zip((path_ds, feat_ds, label_ds))

    def preprocess(path, feat, label):
        img = load_image(path)
        if training:
            img = data_augmentation(img, training=True)
        return (img, feat), label

    ds = ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.shuffle(1024)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

    if training:
        def mixup_batch(images, features, labels):
            batch_size = tf.shape(images)[0]
            indices = tf.random.shuffle(tf.range(batch_size))
            lam = tf.random.uniform(shape=[batch_size, 1, 1, 1], minval=0.0, maxval=1.0)
            lam = tf.maximum(lam, 1 - MIXUP_ALPHA)

            mixed_images = lam * images + (1. - lam) * tf.gather(images, indices)

            lam_feat = lam[:, 0, 0, 0:1]  # shape (batch,1)
            mixed_features = lam_feat * features + (1. - lam_feat) * tf.gather(features, indices)

            labels_onehot = tf.one_hot(labels, NUM_CLASSES)
            labels_shuffled = tf.gather(labels_onehot, indices)
            lam_lab = lam_feat
            mixed_labels = lam_lab * labels_onehot + (1. - lam_lab) * labels_shuffled

            return (mixed_images, mixed_features), mixed_labels

        # ✅ Correct mapping: unpack the nested ((images, features), labels)
        ds = ds.map(lambda img_feat, lbls: mixup_batch(img_feat[0], img_feat[1], lbls),
                    num_parallel_calls=tf.data.AUTOTUNE)
    else:
        ds = ds.map(lambda imgs_feats, lbls: ((imgs_feats[0], imgs_feats[1]), tf.one_hot(lbls, NUM_CLASSES)),
                    num_parallel_calls=tf.data.AUTOTUNE)

    return ds

train_ds = create_hybrid_dataset(paths_train2, X_hand_train2, y_train2, training=True)
val_ds   = create_hybrid_dataset(paths_val2, X_hand_val2, y_val2, training=False)
test_ds  = create_hybrid_dataset(paths_test, X_hand_test, y_test, training=False)

# ---------- 5. Model architecture (increased capacity) ----------
def se_block(x, ratio=16):
    filters = x.shape[-1]
    se = layers.GlobalAveragePooling2D()(x)
    se = layers.Dense(filters // ratio, activation='relu')(se)
    se = layers.Dense(filters, activation='sigmoid')(se)
    return layers.Multiply()([x, se])

# CNN stem (filters: 32 → 64 → 192)
img_input = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='image_input')
x = layers.Conv2D(32, (3,3), activation='relu', padding='same')(img_input)
x = layers.MaxPooling2D((2,2))(x)
x = layers.BatchNormalization()(x)
x = se_block(x)

x = layers.Conv2D(64, (3,3), activation='relu', padding='same')(x)
x = layers.MaxPooling2D((2,2))(x)
x = layers.BatchNormalization()(x)
x = se_block(x)

x = layers.Conv2D(192, (3,3), activation='relu', padding='same')(x)   # increased
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(192, activation='relu', name='img_features')(x)      # increased
x = layers.Dropout(0.3)(x)

# Handcrafted input
feat_input = layers.Input(shape=(252,), name='feat_input')

# ---- Gated Cross‑Modal Fusion ----
img_gate_proj = layers.Dense(252, activation='sigmoid')(x)
gated_feat = layers.Multiply()([feat_input, img_gate_proj])

feat_gate_proj = layers.Dense(192, activation='sigmoid')(feat_input)
gated_img_feat = layers.Multiply()([x, feat_gate_proj])

concat = layers.Concatenate()([gated_img_feat, gated_feat])

# Fusion head (wider)
z = layers.Dense(256, activation='relu')(concat)
z = layers.BatchNormalization()(z)
z = layers.Dropout(0.5)(z)                     # slightly higher dropout
z = layers.Dense(128, activation='relu')(z)
z = layers.BatchNormalization()(z)
z = layers.Dropout(0.5)(z)

output = layers.Dense(NUM_CLASSES, activation='softmax')(z)
model = models.Model(inputs=[img_input, feat_input], outputs=output)

# ---------- 6. Cosine decay schedule ----------
steps_per_epoch = tf.math.ceil(len(paths_train2) / BATCH_SIZE)
cosine_scheduler = tf.keras.optimizers.schedules.CosineDecayRestarts(
    initial_learning_rate=1e-3,
    first_decay_steps=int(steps_per_epoch * 10),   # restart every 10 epochs
    t_mul=2.0,
    m_mul=0.9,
    alpha=1e-6
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=cosine_scheduler),
    loss='categorical_crossentropy',   # works with one-hot labels
    metrics=['accuracy']
)
model.summary()

# ---------- 7. Callbacks ----------
early_stop = callbacks.EarlyStopping(
    monitor='val_accuracy', patience=30, restore_best_weights=True
)

class PlotHistory(callbacks.Callback):
    def on_train_end(self, logs=None):
        hist = self.model.history
        # Accuracy curve
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
        ax1.plot(hist.history['accuracy'], label='Train')
        ax1.plot(hist.history['val_accuracy'], label='Validation')
        ax1.set_title('Accuracy over Epochs (V2)')
        ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
        ax1.legend()
        # Loss curve
        ax2.plot(hist.history['loss'], label='Train')
        ax2.plot(hist.history['val_loss'], label='Validation')
        ax2.set_title('Loss over Epochs (V2)')
        ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
        ax2.legend()
        plt.tight_layout()
        plt.savefig('/kaggle/working/accuracy_loss_curves_v2.png', dpi=150)
        plt.close()

# ---------- 8. Train ----------
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[early_stop, PlotHistory()],
    verbose=1
)

# ---------- 9. Standard evaluation (without TTA) ----------
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"\n✅ Hybrid Fusion Model V2 Test Accuracy (standard): {test_acc:.4f}")

y_pred_probs_onehot = model.predict(test_ds)
y_pred = np.argmax(y_pred_probs_onehot, axis=1)

# Classification report (use original sparse labels for clarity)
print("\nClassification Report (standard):")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=4))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix – V2 (standard)')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.tight_layout()
plt.savefig('/kaggle/working/hybrid_fusion_cm_v2_standard.png', dpi=150)
plt.close()

# ---------- 10. Test-Time Augmentation (TTA) ----------
def predict_with_tta(model, test_ds):
    """Average predictions over original + horizontally flipped images."""
    all_probs = []
    for (img_batch, feat_batch), _ in test_ds:
        # original
        p1 = model.predict([img_batch, feat_batch], verbose=0)
        # flipped images
        img_flip = tf.image.flip_left_right(img_batch)
        p2 = model.predict([img_flip, feat_batch], verbose=0)
        # average
        p_avg = (p1 + p2) / 2.0
        all_probs.append(p_avg)
    return np.concatenate(all_probs, axis=0)

y_pred_probs_tta = predict_with_tta(model, test_ds)
y_pred_tta = np.argmax(y_pred_probs_tta, axis=1)

# TTA evaluation (accuracy manually)
tta_acc = np.mean(y_pred_tta == y_test)
print(f"\n✅ Hybrid Fusion Model V2 Test Accuracy (with TTA): {tta_acc:.4f}")

print("\nClassification Report (TTA):")
print(classification_report(y_test, y_pred_tta, target_names=CLASS_NAMES, digits=4))

# Confusion matrix (TTA)
cm_tta = confusion_matrix(y_test, y_pred_tta)
plt.figure(figsize=(6,5))
sns.heatmap(cm_tta, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix – V2 (TTA)')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.tight_layout()
plt.savefig('/kaggle/working/hybrid_fusion_cm_v2_tta.png', dpi=150)
plt.close()

# ---------- 11. ROC curves (per class, using TTA probabilities) ----------
y_test_bin = label_binarize(y_test, classes=range(NUM_CLASSES))
fpr = dict(); tpr = dict(); roc_auc = dict()
for i in range(NUM_CLASSES):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_pred_probs_tta[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

plt.figure(figsize=(8,6))
for i in range(NUM_CLASSES):
    plt.plot(fpr[i], tpr[i],
             label=f'{CLASS_NAMES[i]} (AUC = {roc_auc[i]:.3f})')
plt.plot([0,1], [0,1], 'k--')
plt.xlim([0.0, 1.0]); plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves per Class – V2 (TTA)')
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig('/kaggle/working/roc_curves_v2.png', dpi=150)
plt.close()

# ---------- 12. Save model ----------
model.save(OUTPUT_MODEL)
print(f"Model saved to {OUTPUT_MODEL}")

In [ ]:
# ============================================================================
# CELL 7 : FEATURE SELECTION (ANOVA → 50, Chi2 → 40, RF → 50)
# ============================================================================
from sklearn.feature_selection import SelectKBest, f_classif, chi2
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
import numpy as np
import os

# Load data
#df = pd.read_csv(os.path.join(OUTPUT_DIR, 'features_252_from_overlays.csv'))
X = df.drop(columns=['label', 'image_path']).values
y = df['label'].values
feature_names = df.drop(columns=['label', 'image_path']).columns

# ----- 1. ANOVA F-measure (select top 50) -----
selector_anova = SelectKBest(score_func=f_classif, k=50)
X_anova = selector_anova.fit_transform(X, y)
anova_mask = selector_anova.get_support()
anova_selected = feature_names[anova_mask].tolist()
print(f"ANOVA selected {len(anova_selected)} features")

# ----- 2. Chi-squared test (select top 40) -----
# Chi2 requires non-negative features, so we scale to [0,1]
mms = MinMaxScaler()
X_chi2 = mms.fit_transform(X)
selector_chi2 = SelectKBest(score_func=chi2, k=40)
X_chi2 = selector_chi2.fit_transform(X_chi2, y)
chi2_mask = selector_chi2.get_support()
chi2_selected = feature_names[chi2_mask].tolist()
print(f"Chi2 selected {len(chi2_selected)} features")

# ----- 3. Random Forest importance (select top 50) -----
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X, y)
importances = rf.feature_importances_
rf_indices = np.argsort(importances)[::-1][:50]   # top 50
rf_selected = feature_names[rf_indices].tolist()
print(f"RF selected {len(rf_selected)} features")

# Save the reduced datasets
df_anova = pd.DataFrame(X[:, anova_mask], columns=anova_selected)
df_anova['label'] = y
df_anova.to_csv(os.path.join(OUTPUT_DIR, 'selected_anova_50.csv'), index=False)

df_chi2 = pd.DataFrame(X[:, chi2_mask], columns=chi2_selected)
df_chi2['label'] = y
df_chi2.to_csv(os.path.join(OUTPUT_DIR, 'selected_chi2_40.csv'), index=False)

df_rf = pd.DataFrame(X[:, rf_indices], columns=rf_selected)
df_rf['label'] = y
df_rf.to_csv(os.path.join(OUTPUT_DIR, 'selected_rf_50.csv'), index=False)

print("All feature-selected datasets saved.")

In [ ]:
# ============================================================================
# CELL : TRAIN & EVALUATE ANN ON ALL FEATURE SETS (FIXED)
# ============================================================================
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

# ---------- CONFIG ----------
FEATURES_DIR = '/kaggle/working/feature_analysis'
RESULTS_DIR  = os.path.join(FEATURES_DIR, 'classification_results')
os.makedirs(RESULTS_DIR, exist_ok=True)

CLASS_NAMES = ['Healthy', 'Yellow', 'Cercospora Leaf Spot', 'Bacterial Leaf Spot']

# Map method name → CSV filename
method_files = {
    'Original 252 Features': 'features_252_from_overlays.csv',
    'PCA (70)':              'reduced_pca_70.csv',
    'KPCA (65)':             'reduced_kpca_65.csv',
    'Sparse AE (60)':        'reduced_sparse_ae_60.csv',
    'Stacked AE (126)':      'reduced_stacked_ae_126.csv',
    'ANOVA F-measure (50)':  'selected_anova_50.csv',
    'Chi-square (40)':       'selected_chi2_40.csv',
    'Random Forest (50)':    'selected_rf_50.csv'
}

# ANN hyperparameters
EPOCHS = 100
BATCH_SIZE = 32
VALIDATION_SPLIT = 0.2
RANDOM_STATE = 42

# ============================================================================
# Helper: build simple 2‑hidden‑layer MLP
# ============================================================================
def build_ann(input_dim, n_classes=4):
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(128, activation='relu'),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(n_classes, activation='softmax')
    ])
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# ============================================================================
# Loop over all methods
# ============================================================================
summary_results = {}

for method_name, csv_file in method_files.items():
    csv_path = os.path.join(FEATURES_DIR, csv_file)
    if not os.path.exists(csv_path):
        print(f"⚠️  {csv_file} not found. Skipping '{method_name}'.")
        continue

    print(f"\n{'='*60}")
    print(f"Processing: {method_name}")
    print(f"{'='*60}")

    # 1. Load data
    df = pd.read_csv(csv_path)

    # Drop label and any non-numeric columns (like 'image_path')
    df_numeric = df.drop(columns=['label'], errors='ignore')
    # Also drop 'image_path' if it exists
    df_numeric = df_numeric.drop(columns=['image_path'], errors='ignore')
    # Optionally drop any other non-numeric columns (safety)
    df_numeric = df_numeric.select_dtypes(include=[np.number])

    X = df_numeric.values
    y = df['label'].values

    # 2. Train/validation split (stratified)
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=VALIDATION_SPLIT, stratify=y, random_state=RANDOM_STATE
    )

    # 3. Standardize
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val   = scaler.transform(X_val)

    # 4. One‑hot encode labels
    y_train_cat = to_categorical(y_train, num_classes=4)
    y_val_cat   = to_categorical(y_val, num_classes=4)

    # 5. Build and train model
    model = build_ann(input_dim=X_train.shape[1])
    early_stop = EarlyStopping(monitor='val_accuracy', patience=20,
                               restore_best_weights=True, verbose=0)

    history = model.fit(
        X_train, y_train_cat,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_val, y_val_cat),
        callbacks=[early_stop],
        verbose=0
    )

    # 6. Evaluate on validation set
    y_pred_probs = model.predict(X_val)
    y_pred = np.argmax(y_pred_probs, axis=1)

    # 7. Classification report (save as text)
    report = classification_report(y_val, y_pred, target_names=CLASS_NAMES, digits=4)
    print(report)

    # Save report to file
    report_path = os.path.join(RESULTS_DIR, f'{method_name.replace(" ", "_")}_report.txt')
    with open(report_path, 'w') as f:
        f.write(report)

    # 8. Confusion matrix (save figure)
    cm = confusion_matrix(y_val, y_pred)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f'Confusion Matrix - {method_name}')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    cm_path = os.path.join(RESULTS_DIR, f'{method_name.replace(" ", "_")}_cm.png')
    plt.savefig(cm_path, dpi=150)
    plt.close()

    # 9. Training curves (accuracy & loss)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12,4))
    # Accuracy
    ax1.plot(history.history['accuracy'], label='Train Accuracy')
    ax1.plot(history.history['val_accuracy'], label='Val Accuracy')
    ax1.set_title(f'Accuracy - {method_name}')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    # Loss
    ax2.plot(history.history['loss'], label='Train Loss')
    ax2.plot(history.history['val_loss'], label='Val Loss')
    ax2.set_title(f'Loss - {method_name}')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    plt.tight_layout()
    curves_path = os.path.join(RESULTS_DIR, f'{method_name.replace(" ", "_")}_curves.png')
    plt.savefig(curves_path, dpi=150)
    plt.close()

    # 10. Store metrics for summary
    val_acc = max(history.history['val_accuracy'])
    val_loss = min(history.history['val_loss'])
    summary_results[method_name] = {
        'val_accuracy': val_acc,
        'val_loss': val_loss,
        'report_path': report_path,
        'cm_path': cm_path,
        'curves_path': curves_path
    }

# 11. Save summary as JSON for easy comparison
summary_path = os.path.join(RESULTS_DIR, 'summary_metrics.json')
with open(summary_path, 'w') as f:
    json.dump({k: {'val_accuracy': v['val_accuracy'], 'val_loss': v['val_loss']}
               for k, v in summary_results.items()}, f, indent=4)

print(f"\n✅ All experiments completed. Results saved to: {RESULTS_DIR}")
print("Summary metrics:\n", summary_path)

In [ ]:
# ============================================================================
# CELL: BOOST ACCURACY – XGBOOST + ENSEMBLE (run after all CSVs exist)
# ============================================================================
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier  # alternative ANN with more options

FEATURES_DIR = '/kaggle/working/feature_analysis'
CLASS_NAMES = ['Healthy','Yellow','Cercospora Leaf Spot','Bacterial Leaf Spot']

# Map of the best performing feature sets (can add more)
best_sets = {
    'Original 252': 'features_252_from_overlays.csv',
    'RF-50':        'selected_rf_50.csv',
    'PCA-70':       'reduced_pca_70.csv',
    'StackedAE-126':'reduced_stacked_ae_126.csv'
}

# Load all and store as (name, X_scaled, y)
datasets = {}
for name, fname in best_sets.items():
    path = os.path.join(FEATURES_DIR, fname)
    if not os.path.exists(path):
        continue
    df = pd.read_csv(path)
    X = df.drop(columns=['label', 'image_path'], errors='ignore').select_dtypes(include=[np.number]).values
    y = df['label'].values
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    datasets[name] = (X_scaled, y)

print("Datasets loaded:", list(datasets.keys()))

# =============================================================================
# 1. Train XGBoost on each dataset and evaluate with 5-fold CV
# =============================================================================
print("\n--- XGBoost 5‑fold CV accuracy ---")
xgb = XGBClassifier(objective='multi:softmax', num_class=4,
                    eval_metric='mlogloss', use_label_encoder=False,
                    random_state=42, n_jobs=-1)
for name, (X, y) in datasets.items():
    scores = cross_val_score(xgb, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=42),
                             scoring='accuracy')
    print(f"{name:20s}: {scores.mean():.4f} ± {scores.std():.4f}")

# =============================================================================
# 2. Build a voting ensemble from the strongest models
#    (here we build a soft‑voting classifier using XGBoost, Random Forest, and a tuned MLP)
# =============================================================================
print("\n--- Voting Ensemble (on Original 252 features) ---")
X_orig, y_orig = datasets['Original 252']

# Base classifiers
clf1 = XGBClassifier(objective='multi:softmax', num_class=4, eval_metric='mlogloss',
                     use_label_encoder=False, random_state=42, n_jobs=-1)
clf2 = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
clf3 = MLPClassifier(hidden_layer_sizes=(256, 128, 64), activation='relu',
                     max_iter=300, random_state=42)

ensemble = VotingClassifier(estimators=[('xgb', clf1), ('rf', clf2), ('mlp', clf3)],
                            voting='soft')
scores_ens = cross_val_score(ensemble, X_orig, y_orig,
                             cv=StratifiedKFold(5, shuffle=True, random_state=42),
                             scoring='accuracy')
print(f"Voting Ensemble mean accuracy: {scores_ens.mean():.4f}")

# =============================================================================
# 3. (Optional) Hyper-parameter tuning for XGBoost with GridSearchCV
# =============================================================================
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 1.0]
}
grid = GridSearchCV(XGBClassifier(objective='multi:softmax', num_class=4,
                                  eval_metric='mlogloss', use_label_encoder=False,
                                  random_state=42, n_jobs=-1),
                    param_grid, cv=3, scoring='accuracy', verbose=1)
grid.fit(X_orig, y_orig)
print(f"Best XGBoost params: {grid.best_params_} | Accuracy: {grid.best_score_:.4f}")

print("\n✅ All boosting experiments complete.")

In [ ]:
# ============================================================================
# CELL: BOOST ACCURACY – COMBINE FEATURES + OPTUNA-TUNED DNN
# ============================================================================
import os, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.decomposition import PCA
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
import optuna

# ---------- CONFIG ----------
FEATURES_DIR = '/kaggle/working/feature_analysis'
OUTPUT_DIR   = '/kaggle/working/boosted_model'
os.makedirs(OUTPUT_DIR, exist_ok=True)

CLASS_NAMES = ['Healthy', 'Yellow', 'Cercospora Leaf Spot', 'Bacterial Leaf Spot']
RANDOM_SEED = 42
N_CLASSES = len(CLASS_NAMES)

# ---------- 1. Load & combine feature sets ----------
# Best performing sets (adjust names if needed)
sets_to_combine = {
    'Original 252': 'features_252_from_overlays.csv',
    'RF-50':        'selected_rf_50.csv',
    'PCA-70':       'reduced_pca_70.csv',
    'StackedAE-126':'reduced_stacked_ae_126.csv'
}

all_X = []
for name, fname in sets_to_combine.items():
    path = os.path.join(FEATURES_DIR, fname)
    if not os.path.exists(path):
        print(f"Warning: {fname} missing, skipping.")
        continue
    df = pd.read_csv(path)
    X = df.drop(columns=['label', 'image_path'], errors='ignore').select_dtypes(include=[np.number])
    # standardize each set independently
    scaler = StandardScaler()
    X_std = scaler.fit_transform(X)
    all_X.append(X_std)
    print(f"{name} loaded: {X_std.shape}")

# Stack horizontally
X_combined = np.hstack(all_X)
print(f"Combined feature shape: {X_combined.shape}")

# Use the labels from the first CSV (all share same order)
df_labels = pd.read_csv(os.path.join(FEATURES_DIR, 'features_252_from_overlays.csv'))
y = df_labels['label'].values

# ---------- 2. Reduce dimensionality with PCA ----------
# Keep 180 components – retains >95% variance usually
pca = PCA(n_components=180, random_state=RANDOM_SEED)
X_pca = pca.fit_transform(X_combined)
expl_var = pca.explained_variance_ratio_.sum()
print(f"Explained variance with 180 PCs: {expl_var:.3f}")

# ---------- 3. Train / Test split (stratified) ----------
X_train, X_test, y_train, y_test = train_test_split(
    X_pca, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED
)

# ---------- 4. Optuna hyper-parameter tuning ----------
def create_model(trial):
    # Architecture
    n_layers = trial.suggest_int('n_layers', 2, 4)
    units = [trial.suggest_int(f'units_{i}', 64, 512, step=64) for i in range(n_layers)]
    dropout = trial.suggest_float('dropout', 0.2, 0.5, step=0.1)
    learning_rate = trial.suggest_float('lr', 1e-4, 1e-2, log=True)

    model = Sequential()
    model.add(Dense(units[0], activation='relu', input_shape=(X_train.shape[1],)))
    model.add(BatchNormalization())
    model.add(Dropout(dropout))

    for u in units[1:]:
        model.add(Dense(u, activation='relu'))
        model.add(BatchNormalization())
        model.add(Dropout(dropout))

    model.add(Dense(N_CLASSES, activation='softmax'))
    model.compile(optimizer=Adam(learning_rate=learning_rate),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

def objective(trial):
    # 5-fold cross validation on TRAINING set only
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    scores = []

    for train_idx, val_idx in skf.split(X_train, y_train):
        X_tr, X_val = X_train[train_idx], X_train[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]
        y_tr_cat = to_categorical(y_tr, N_CLASSES)
        y_val_cat = to_categorical(y_val, N_CLASSES)

        model = create_model(trial)
        es = EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=0)
        history = model.fit(X_tr, y_tr_cat,
                            validation_data=(X_val, y_val_cat),
                            epochs=100, batch_size=32,
                            callbacks=[es], verbose=0)
        scores.append(max(history.history['val_accuracy']))

    return np.mean(scores)

# Run Optuna study (set n_trials as needed – 20‑30 for decent tuning)
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30, show_progress_bar=True)
print("Best trial:")
best_trial = study.best_trial
print(f"  Accuracy: {best_trial.value:.4f}")
print(f"  Params: {best_trial.params}")

# ---------- 5. Train final model with best params on (X_train, y_train) ----------
# Also reserve a validation part from X_train to get curves
X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=RANDOM_SEED
)

best_model = create_model(study.best_trial)
y_train_final_cat = to_categorical(y_train_final, N_CLASSES)
y_val_cat = to_categorical(y_val, N_CLASSES)

reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)
early_stop = EarlyStopping(monitor='val_accuracy', patience=20, restore_best_weights=True, verbose=1)

history = best_model.fit(
    X_train_final, y_train_final_cat,
    validation_data=(X_val, y_val_cat),
    epochs=200, batch_size=32,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

# ---------- 6. Evaluate on held‑out TEST set ----------
y_test_cat = to_categorical(y_test, N_CLASSES)
y_pred_probs = best_model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

# Classification report
report = classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=4)
print("\n=== Classification Report (Test Set) ===")
print(report)
with open(os.path.join(OUTPUT_DIR, 'classification_report.txt'), 'w') as f:
    f.write(report)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix – Tuned DNN')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
cm_path = os.path.join(OUTPUT_DIR, 'confusion_matrix.png')
plt.savefig(cm_path, dpi=150)
plt.close()

# Accuracy & Loss curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12,4))
ax1.plot(history.history['accuracy'], label='Train Accuracy')
ax1.plot(history.history['val_accuracy'], label='Val Accuracy')
ax1.set_title('Accuracy Curves')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()

ax2.plot(history.history['loss'], label='Train Loss')
ax2.plot(history.history['val_loss'], label='Val Loss')
ax2.set_title('Loss Curves')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
plt.tight_layout()
curves_path = os.path.join(OUTPUT_DIR, 'accuracy_loss_curves.png')
plt.savefig(curves_path, dpi=150)
plt.close()

# Save final model and PCA
best_model.save(os.path.join(OUTPUT_DIR, 'best_model.h5'))
import joblib
joblib.dump(pca, os.path.join(OUTPUT_DIR, 'pca_transformer.pkl'))

print(f"\n✅ Boosted model results saved to: {OUTPUT_DIR}")
print("Final test accuracy (hold-out):", (y_pred == y_test).mean())

In [ ]:
!pip install lightgbm -q

In [ ]:
# ============================================================================
# CELL: STACKING ENSEMBLE (XGBoost + LightGBM + MLP) – CORRECTED
# ============================================================================
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.base import BaseEstimator, TransformerMixin

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

import xgboost as xgb
import lightgbm as lgb

warnings.filterwarnings('ignore')

# ---------- CONFIG ----------
FEATURES_DIR = '/kaggle/working/feature_analysis'
OUTPUT_DIR   = '/kaggle/working/stacking_ensemble'
os.makedirs(OUTPUT_DIR, exist_ok=True)

CLASS_NAMES = ['Healthy', 'Yellow', 'Cercospora Leaf Spot', 'Bacterial Leaf Spot']
N_CLASSES = len(CLASS_NAMES)
RANDOM_SEED = 42
CV_FOLDS = 5

# ---------- 1. Load & combine feature sets ----------
# ---------- 1. Load & combine feature sets ----------
feature_sets = {
    'Original 252': 'features_252_from_overlays.csv',
    'RF-50':        'selected_rf_50.csv',
    'PCA-70':       'reduced_pca_70.csv',
    'StackedAE-126':'reduced_stacked_ae_126.csv'
}

all_X = []
for name, fname in feature_sets.items():
    path = os.path.join(FEATURES_DIR, fname)
    if not os.path.exists(path):
        continue
    df = pd.read_csv(path)
    X = df.drop(columns=['label', 'image_path'], errors='ignore').select_dtypes(include=[np.number])
    scaler = StandardScaler()
    X_std = scaler.fit_transform(X)
    all_X.append(X_std)
    print(f"{name}: {X_std.shape[1]} features")

X_combined = np.hstack(all_X)

# Load labels properly – use the first CSV and the 'label' column
df_labels = pd.read_csv( os.path.join(FEATURES_DIR, 'features_252_from_overlays.csv') )
y = df_labels['label'].values.astype(int)   # ensure integer

print(f"Combined feature shape: {X_combined.shape}")
print("Class distribution in full data:", np.bincount(y))


# ---------- 2. Train / Test split ----------
X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED
)
print("Training set class distribution:", np.bincount(y_train))

# ---------- 3. Build stacking ensemble ----------
class KerasClassifierWrapper(BaseEstimator, TransformerMixin):
    def __init__(self, build_fn, epochs=100, batch_size=32, verbose=0):
        self.build_fn = build_fn
        self.epochs = epochs
        self.batch_size = batch_size
        self.verbose = verbose

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.model_ = self.build_fn(X.shape[1], len(self.classes_))
        y_cat = to_categorical(y, len(self.classes_))
        es = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True, verbose=0)
        self.model_.fit(X, y_cat, epochs=self.epochs, batch_size=self.batch_size,
                        callbacks=[es], verbose=self.verbose)
        return self

    def predict(self, X):
        return np.argmax(self.model_.predict(X), axis=1)

    def predict_proba(self, X):
        return self.model_.predict(X)

def build_mlp(input_dim, n_classes):
    model = Sequential([
        Dense(512, activation='relu', input_shape=(input_dim,)),
        BatchNormalization(),
        Dropout(0.4),
        Dense(256, activation='relu'),
        BatchNormalization(),
        Dropout(0.4),
        Dense(128, activation='relu'),
        BatchNormalization(),
        Dropout(0.4),
        Dense(n_classes, activation='softmax')
    ])
    model.compile(optimizer=Adam(learning_rate=0.001),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# Base models
xgb_clf = xgb.XGBClassifier(
    objective='multi:softprob', num_class=N_CLASSES,
    eval_metric='mlogloss', use_label_encoder=False,
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_SEED
)

lgb_clf = lgb.LGBMClassifier(
    objective='multiclass', num_class=N_CLASSES,
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_SEED,
    verbose=-1
)

mlp_clf = KerasClassifierWrapper(build_fn=build_mlp, epochs=100, batch_size=32, verbose=0)

base_models = [
    ('xgb', xgb_clf),
    ('lgb', lgb_clf),
    ('mlp', mlp_clf)
]

meta_clf = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=500, C=5.0, random_state=RANDOM_SEED)

from sklearn.ensemble import StackingClassifier

stacking_clf = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_clf,
    cv=CV_FOLDS,
    stack_method='predict_proba',
    n_jobs=-1
)

# Fit stacking classifier
stacking_clf.fit(X_train, y_train)

# ---------- 4. Evaluate on test set ----------
y_pred = stacking_clf.predict(X_test)
test_acc = np.mean(y_pred == y_test)
print(f"\n=== Stacking Test Accuracy: {test_acc:.4f} ===")

report = classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=4)
print(report)
with open(os.path.join(OUTPUT_DIR, 'classification_report.txt'), 'w') as f:
    f.write(report)

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix – Stacking Ensemble')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
cm_path = os.path.join(OUTPUT_DIR, 'confusion_matrix.png')
plt.savefig(cm_path, dpi=150)
plt.close()

# ---------- 5. MLP curves (without stratification) ----------
X_train_mlp, X_val_mlp, y_train_mlp, y_val_mlp = train_test_split(
    X_train, y_train, test_size=0.2, random_state=RANDOM_SEED   # <-- no stratify
)
y_train_mlp_cat = to_categorical(y_train_mlp, N_CLASSES)
y_val_mlp_cat = to_categorical(y_val_mlp, N_CLASSES)

mlp_for_curves = build_mlp(X_train.shape[1], N_CLASSES)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6, verbose=0)
early_stop = EarlyStopping(monitor='val_accuracy', patience=30, restore_best_weights=True, verbose=0)
history_mlp = mlp_for_curves.fit(
    X_train_mlp, y_train_mlp_cat,
    validation_data=(X_val_mlp, y_val_mlp_cat),
    epochs=200, batch_size=32,
    callbacks=[early_stop, reduce_lr],
    verbose=0
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12,4))
ax1.plot(history_mlp.history['accuracy'], label='Train Accuracy')
ax1.plot(history_mlp.history['val_accuracy'], label='Val Accuracy')
ax1.set_title('MLP Accuracy Curves')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()

ax2.plot(history_mlp.history['loss'], label='Train Loss')
ax2.plot(history_mlp.history['val_loss'], label='Val Loss')
ax2.set_title('MLP Loss Curves')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
plt.tight_layout()
curves_path = os.path.join(OUTPUT_DIR, 'accuracy_loss_curves_mlp.png')
plt.savefig(curves_path, dpi=150)
plt.close()

import joblib
joblib.dump(stacking_clf, os.path.join(OUTPUT_DIR, 'stacking_classifier.pkl'))

print(f"\n✅ All results saved to: {OUTPUT_DIR}")

In [ ]:
# ============================================================================
# CELL: ZIP THE FEATURE ANALYSIS FOLDER FOR DOWNLOAD
# ============================================================================
import shutil
import os

FEATURES_DIR = '/kaggle/working/'  # folder to zip
ZIP_PATH     = '/kaggle/working/final_codeResult.zip'  # output zip file

# Create zip archive
shutil.make_archive(ZIP_PATH.replace('.zip', ''), 'zip', FEATURES_DIR)

print(f"✅ Done! Download the zip from:\n{ZIP_PATH}")

## CNN and XAI

In [ ]:
# ============================================================================
# CELL: TRAIN CNN ON SEGMENTED OVERLAY IMAGES (for XAI)
# ============================================================================
import os, json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt

# ---------- CONFIG ----------
IMAGE_DIR = '/kaggle/input/datasets/sabitahamedpreanto/moringa-segmented/overlays'
IMG_SIZE  = 256
BATCH     = 32
EPOCHS    = 30
SEED      = 42
CLASS_NAMES = ['Healthy', 'Yellow', 'Cercospora Leaf Spot', 'Bacterial Leaf Spot']

# ---------- Load image paths and labels ----------
paths, labels = [], []
for idx, cls in enumerate(CLASS_NAMES):
    for fname in os.listdir(IMAGE_DIR):
        if fname.startswith(cls + '__'):
            paths.append(os.path.join(IMAGE_DIR, fname))
            labels.append(idx)

X_paths, y = np.array(paths), np.array(labels)

# Shuffle and split (keeping 20% for validation)
train_paths, val_paths, y_train, y_val = train_test_split(
    X_paths, y, test_size=0.2, stratify=y, random_state=SEED
)

# ---------- Keras data generators ----------
def load_img(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=3)  # overlay saved as PNG
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0
    return img

train_ds = tf.data.Dataset.from_tensor_slices((train_paths, y_train))
train_ds = train_ds.shuffle(1000).map(lambda p, l: (load_img(p), l),
                                      num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((val_paths, y_val))
val_ds = val_ds.map(lambda p, l: (load_img(p), l),
                    num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH).prefetch(tf.data.AUTOTUNE)

# ---------- Simple CNN model ----------
def create_cnn():
    model = models.Sequential([
        layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
        layers.Conv2D(32, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.GlobalAveragePooling2D(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(4, activation='softmax')
    ])
    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

cnn_model = create_cnn()
cnn_model.summary()

# ---------- Train ----------
early_stop = callbacks.EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True)
history = cnn_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS,
                        callbacks=[early_stop], verbose=1)

# ---------- Save model ----------
cnn_model.save('/kaggle/working/cnn_overlay_model.h5')
print("CNN model saved to /kaggle/working/cnn_overlay_model.h5")

In [ ]:
# ============================================================================
# CELL: INSTALL XAI LIBRARIES (run once)
# ============================================================================
!pip install tf-keras-vis shap lime --quiet

In [ ]:
# ============================================================================
# CELL: XAI FOR CNN (Grad-CAM, Grad-CAM++, Score-CAM, LIME, SHAP)
# ============================================================================
import os, random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import skimage.segmentation

# XAI imports
from tf_keras_vis.utils.scores import CategoricalScore
from tf_keras_vis.gradcam import Gradcam, GradcamPlusPlus
from tf_keras_vis.scorecam import Scorecam
import shap
import lime
from lime import lime_image

# ---------- Config ----------
MODEL_PATH = '/kaggle/working/cnn_overlay_model.h5'
OVERLAY_DIR = '/kaggle/input/datasets/sabitahamedpreanto/moringa-segmented/overlays'
OUTPUT_XAI_DIR = '/kaggle/working/xai_cnn_results'
os.makedirs(OUTPUT_XAI_DIR, exist_ok=True)

CLASS_NAMES = ['Healthy', 'Yellow', 'Cercospora Leaf Spot', 'Bacterial Leaf Spot']
IMG_SIZE = 256

# ---------- Load model and force build ----------
model = load_model(MODEL_PATH, compile=True)
# Force build by calling model on a dummy input
_ = model(tf.zeros((1, IMG_SIZE, IMG_SIZE, 3)))   # builds the graph
print("Model built successfully")
model.summary()

# Find the last Dense layer with 4 units (logits)
logits_layer = None
for layer in reversed(model.layers):
    if isinstance(layer, tf.keras.layers.Dense) and layer.units == 4:
        logits_layer = layer
        break
if logits_layer is None:
    raise ValueError("Could not find a Dense layer with 4 units (logits). Check model architecture.")

# Create logits model - use model.inputs[0] (now defined)
logits_model = tf.keras.Model(inputs=model.inputs[0], outputs=logits_layer.output)
print(f"Logits model created using layer: {logits_layer.name}")

# ---------- Prepare one image per class ----------
def load_preprocess_img(path):
    img = image.load_img(path, target_size=(IMG_SIZE, IMG_SIZE))
    img = image.img_to_array(img) / 255.0
    return img

# Collect one image per class (files named like "Healthy__xxx.jpg")
sample_paths = []
for cls_id, cls_name in enumerate(CLASS_NAMES):
    candidates = [f for f in os.listdir(OVERLAY_DIR) if f.startswith(cls_name + '__')]
    if not candidates:
        print(f"Warning: No image found for class {cls_name}")
        continue
    chosen = random.choice(candidates)
    sample_paths.append((cls_id, os.path.join(OVERLAY_DIR, chosen), cls_name))

if not sample_paths:
    raise FileNotFoundError("No sample images found for any class in OVERLAY_DIR")

# ---------- tf-keras-vis setup ----------
def model_modifier(m):
    m.layers[-1].activation = tf.keras.activations.linear
    return m

gradcam = Gradcam(model, model_modifier=model_modifier, clone=True)
gradcam_pp = GradcamPlusPlus(model, model_modifier=model_modifier, clone=True)
scorecam = Scorecam(model, model_modifier=None)

# ---------- SHAP background (20 random images from overlays) ----------
background = []
image_paths = [os.path.join(OVERLAY_DIR, f) for f in os.listdir(OVERLAY_DIR) 
               if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
if len(image_paths) == 0:
    raise FileNotFoundError(f"No image files found in {OVERLAY_DIR}")

random.shuffle(image_paths)
background_paths = image_paths[:20]

for path in background_paths:
    img = image.load_img(path, target_size=(IMG_SIZE, IMG_SIZE))
    img = image.img_to_array(img) / 255.0
    background.append(img)

background = np.array(background, dtype=np.float32)
print(f"✅ Loaded {len(background)} background images for SHAP")

# ---------- Loop over sample images ----------
for cls_id, img_path, cls_name in sample_paths:
    print(f"\nProcessing {cls_name} ...")
    img = load_preprocess_img(img_path)
    img_batch = np.expand_dims(img, axis=0)

    # Prediction
    pred = model.predict(img_batch, verbose=0)[0]
    pred_class = np.argmax(pred)
    target_class = pred_class
    print(f"  True class: {cls_name} | Predicted: {CLASS_NAMES[pred_class]} (confidence {pred[pred_class]:.3f})")

    # Prepare score for GradCAM etc.
    score = CategoricalScore(target_class)

    # ----- Grad-CAM -----
    try:
        cam = gradcam(score, img_batch, penultimate_layer=-1)
        cam = np.squeeze(cam)
        _, ax = plt.subplots(1,3, figsize=(15,5))
        ax[0].imshow(img)
        ax[0].set_title(f"{cls_name} (pred: {CLASS_NAMES[pred_class]})")
        ax[1].imshow(cam, cmap='jet')
        ax[1].set_title("Grad-CAM")
        ax[2].imshow(img)
        ax[2].imshow(cam, cmap='jet', alpha=0.4)
        ax[2].set_title("Overlay")
        for a in ax: a.axis('off')
        plt.savefig(os.path.join(OUTPUT_XAI_DIR, f'{cls_name}_gradcam.png'), bbox_inches='tight')
        plt.close()
    except Exception as e:
        print(f"  Grad-CAM failed: {e}")

    # ----- Grad-CAM++ -----
    try:
        cam_pp = gradcam_pp(score, img_batch, penultimate_layer=-1)
        cam_pp = np.squeeze(cam_pp)
        _, ax = plt.subplots(1,3, figsize=(15,5))
        ax[0].imshow(img)
        ax[0].set_title(f"{cls_name} original")
        ax[1].imshow(cam_pp, cmap='jet')
        ax[1].set_title("Grad-CAM++")
        ax[2].imshow(img)
        ax[2].imshow(cam_pp, cmap='jet', alpha=0.4)
        ax[2].set_title("Overlay")
        for a in ax: a.axis('off')
        plt.savefig(os.path.join(OUTPUT_XAI_DIR, f'{cls_name}_gradcampp.png'), bbox_inches='tight')
        plt.close()
    except Exception as e:
        print(f"  Grad-CAM++ failed: {e}")

    # ----- Score-CAM -----
    try:
        sc = scorecam(score, img_batch, penultimate_layer=-1)
        sc = np.squeeze(sc)
        _, ax = plt.subplots(1,3, figsize=(15,5))
        ax[0].imshow(img)
        ax[0].set_title("Original")
        ax[1].imshow(sc, cmap='jet')
        ax[1].set_title("Score-CAM")
        ax[2].imshow(img)
        ax[2].imshow(sc, cmap='jet', alpha=0.4)
        ax[2].set_title("Overlay")
        for a in ax: a.axis('off')
        plt.savefig(os.path.join(OUTPUT_XAI_DIR, f'{cls_name}_scorecam.png'), bbox_inches='tight')
        plt.close()
    except Exception as e:
        print(f"  Score-CAM failed: {e}")

    # ----- LIME -----
    try:
        explainer = lime_image.LimeImageExplainer()
        def predict_fn(imgs):
            imgs = imgs.astype(np.float32) / 255.0
            return model.predict(imgs, verbose=0)
        explanation = explainer.explain_instance(
            (img * 255).astype(np.uint8),
            classifier_fn=predict_fn,
            top_labels=1,
            hide_color=0,
            num_samples=500
        )
        temp, mask = explanation.get_image_and_mask(
            target_class,
            positive_only=True,
            num_features=5,
            hide_rest=False
        )
        plt.figure(figsize=(8,8))
        plt.imshow(temp / 255.0)
        plt.title(f"LIME explanation for {cls_name}")
        plt.axis('off')
        plt.savefig(os.path.join(OUTPUT_XAI_DIR, f'{cls_name}_lime.png'), bbox_inches='tight')
        plt.close()
    except Exception as e:
        print(f"  LIME failed: {e}")

    # ----- SHAP (using logits model) -----
    try:
        e = shap.DeepExplainer(logits_model, background)
        shap_values = e.shap_values(img_batch)   # list of arrays, one per class
        # shap_values length = number of classes (4)
        if len(shap_values) != len(CLASS_NAMES):
            print(f"  Warning: SHAP returned {len(shap_values)} value sets, expected {len(CLASS_NAMES)}")
        # Visualise for the predicted class
        class_idx = target_class
        if class_idx < len(shap_values):
            shap_arr = shap_values[class_idx]        # shape (1, 256, 256, 3)
            shap_heatmap = np.abs(shap_arr[0]).max(axis=-1)   # (256,256)
            plt.figure(figsize=(8,8))
            plt.imshow(img)
            plt.imshow(shap_heatmap, cmap='inferno', alpha=0.5)
            plt.title(f"SHAP heatmap (logits) for {cls_name} (class {CLASS_NAMES[class_idx]})")
            plt.axis('off')
            plt.savefig(os.path.join(OUTPUT_XAI_DIR, f'{cls_name}_shap.png'), bbox_inches='tight')
            plt.close()
        else:
            print(f"  SHAP index {class_idx} out of range")
    except Exception as e:
        print(f"  SHAP failed: {e}")

print("\n✅ All CNN XAI visualizations saved to:", OUTPUT_XAI_DIR)

In [ ]:
# ============================================================================
# CELL: ZIP THE FEATURE ANALYSIS FOLDER FOR DOWNLOAD
# ============================================================================
import shutil
import os

FEATURES_DIR = '/kaggle/working/xai_cnn_results'  # folder to zip
ZIP_PATH     = '/kaggle/working/xai_cnn_results.zip'  # output zip file

# Create zip archive
shutil.make_archive(ZIP_PATH.replace('.zip', ''), 'zip', FEATURES_DIR)

print(f"✅ Done! Download the zip from:\n{ZIP_PATH}")